In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
import time

from tqdm import tqdm

import numpy as np
import pandas as pd

import hyperopt.hp as hp

from sklearn.model_selection import train_test_split

from catboost import CatBoostClassifier

from sklift.datasets import fetch_x5
from sklift.models import ClassTransformation, TwoModels
from sklift.metrics import qini_auc_score, uplift_at_k

from causalml.inference.tree import UpliftTreeClassifier as UpliftTreeClassifierCM

from upninja.dml.uplift_tree_dml import UpliftTreeRegressorDML
from upninja.tune.selection import UpliftTune

import matplotlib.pyplot as plt
import seaborn as sns

Failed to import duecredit due to No module named 'duecredit'


In [3]:
SEED = 8

In [4]:
%%time

dataset = fetch_x5()
dataset.data.keys()

CPU times: user 24.3 s, sys: 4.56 s, total: 28.9 s
Wall time: 31.2 s


dict_keys(['clients', 'train', 'purchases'])

In [5]:
%%time

print(f'Dataset type: {type(dataset)}\n')
print(f'Dataset features shape: {dataset.data['clients'].shape}')
print(f'Dataset features shape: {dataset.data['train'].shape}')
print(f'Dataset target shape: {dataset.target.shape}')
print(f'Dataset treatment shape: {dataset.treatment.shape}')

Dataset type: <class 'sklearn.utils._bunch.Bunch'>

Dataset features shape: (400162, 5)
Dataset features shape: (200039, 1)
Dataset target shape: (200039,)
Dataset treatment shape: (200039,)
CPU times: user 62 μs, sys: 50 μs, total: 112 μs
Wall time: 113 μs


In [6]:
%%time

# Извлечение данных
df_clients = dataset.data['clients'].set_index('client_id')
df_train = pd.concat([dataset.data['train'], dataset.treatment , dataset.target], axis=1).set_index('client_id')
indices_test = pd.Index(set(df_clients.index) - set(df_train.index))

# Извлечение признаков
df_features = df_clients.copy()
df_features['first_issue_time'] = \
    (pd.to_datetime(df_features['first_issue_date'])
     - pd.Timestamp('1970-01-01')) // pd.Timedelta('1s')
df_features['first_redeem_time'] = \
    (pd.to_datetime(df_features['first_redeem_date'])
     - pd.Timestamp('1970-01-01')) // pd.Timedelta('1s')
df_features['issue_redeem_delay'] = df_features['first_redeem_time'] \
    - df_features['first_issue_time']
df_features = df_features.drop(['first_issue_date', 'first_redeem_date'], axis=1)

indices_learn, indices_valid = train_test_split(df_train.index, test_size=0.3, random_state=SEED)

CPU times: user 151 ms, sys: 25.2 ms, total: 176 ms
Wall time: 176 ms


In [35]:
%%time

X_train = df_features.loc[indices_learn, :]
y_train = df_train.loc[indices_learn, 'target']
treat_train = df_train.loc[indices_learn, 'treatment_flg']

X_val = df_features.loc[indices_valid, :]
y_val = df_train.loc[indices_valid, 'target']
treat_val =  df_train.loc[indices_valid, 'treatment_flg']

X_train_full = df_features.loc[df_train.index, :]
y_train_full = df_train.loc[:, 'target']
treat_train_full = df_train.loc[:, 'treatment_flg']

X_test = df_features.loc[indices_test, :]

X_train['gender'] = X_train['gender'].map({'F': 0, 'U': -1, 'M': 1})
X_val['gender'] = X_val['gender'].map({'F': 0, 'U': -1, 'M': 1})
X_test['gender'] = X_test['gender'].map({'F': 0, 'U': -1, 'M': 1})

X_train.fillna(-1.0, inplace=True)
X_val.fillna(-1.0, inplace=True)
X_test.fillna(-1.0, inplace=True)

cat_features = ['gender']

CPU times: user 204 ms, sys: 20 ms, total: 224 ms
Wall time: 223 ms


# ✅ Train models

## ⭐ Class Transformation

In [8]:
%%time

cb_params = cb_hp_space = {
    'iterations': hp.uniformint('iterations', 100, 2500),
    'depth': hp.uniformint('depth', 2, 10),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
    'l2_leaf_reg': hp.uniform('l2_leaf_reg', 1, 10),
    'verbose': False,
    'thread_count': -1,
    'random_state': SEED
}

cb_tune = UpliftTune(
    base_model_class=CatBoostClassifier,
    uplift_model_class=ClassTransformation,
    data=X_train,
    target=y_train,
    treatment=treat_train,
    space=cb_params,
    verbose=True,
    max_evals=10
)

t_time_start = time.process_time_ns()
cb_tuned = cb_tune.tune()['best_params']
t_time_end = time.process_time_ns()

100%|███████| 10/10 [02:35<00:00, 15.54s/trial, best loss: -0.04954288228464684]
Optimization completed. Best score: 0.0495
Best parameters: {'depth': 2, 'iterations': 2166, 'l2_leaf_reg': 6.474242459540229, 'learning_rate': 0.06708005741019658, 'random_state': 8, 'thread_count': -1, 'verbose': False}
CPU times: user 21min 14s, sys: 2min 18s, total: 23min 33s
Wall time: 2min 35s


In [9]:
%%time

ct = ClassTransformation(CatBoostClassifier(**cb_tuned))
f_time_start = time.process_time_ns()
ct = ct.fit(X_train, y_train, treat_train, estimator_fit_params={'cat_features': cat_features})
f_time_end = time.process_time_ns()

ct_uplift = ct.predict(X_val)

CPU times: user 3min, sys: 8.79 s, total: 3min 9s
Wall time: 20.8 s


## ⭐ DML Tree

In [10]:
%%time

space = {
    'min_samples': hp.uniformint('min_samples', 20, 500),
    'max_depth': hp.uniformint('max_depth', 3, 20),
    'random_state': SEED
}

dml_tune = UpliftTune(
    uplift_model_class=UpliftTreeRegressorDML,
    data=X_train,
    target=y_train,
    treatment=treat_train,
    space=space,
    verbose=True,
    max_evals=10
)

t_time_start = time.process_time_ns()
dml_tuned = dml_tune.tune()['best_params']
t_time_end = time.process_time_ns()

100%|███████| 10/10 [00:08<00:00,  1.13trial/s, best loss: -0.05889112202442145]
Optimization completed. Best score: 0.0589
Best parameters: {'max_depth': 14, 'min_samples': 411, 'random_state': 8}
CPU times: user 8.73 s, sys: 86 ms, total: 8.81 s
Wall time: 8.82 s


In [11]:
%%time

dml = UpliftTreeRegressorDML(**dml_tuned)

f_time_start = time.process_time_ns()
dml = dml.fit(X_train, y_train, treat_train)
f_time_end = time.process_time_ns()

dml_uplift = dml.predict(X_val)

CPU times: user 525 ms, sys: 4.21 ms, total: 529 ms
Wall time: 529 ms


# ✅ Case

Оценим эффективность назначения коммуникаций при помощи uplift-моделей.

In [18]:
def build_full_roi_table(df, uplift_col, campaign_date='2023-01-01', quantiles=[0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0], sms_cost=8):
    df = df.copy()
    df['transaction_datetime'] = pd.to_datetime(df['transaction_datetime'])
    df['days_since_campaign'] = (df['transaction_datetime'] - pd.to_datetime(campaign_date)).dt.days

    # Фиксированные горизонты
    horizons = {
        '1w': 7,
        '2m': 60,
        '3m': 90,
        '6m': 180,
    }

    results = []

    for horizon_name, max_days in horizons.items():
        # Фильтрация по горизонту
        horizon_df = df[df['days_since_campaign'] <= max_days]
        if horizon_df.empty:
            continue

        # Агрегация по клиенту (сумма покупок, uplift и пр.)
        agg = horizon_df.groupby('client_id').agg({
            'treatment_flg': 'first',
            'target': 'max',  # можно заменить на sum при необходимости
            'purchase_sum': 'sum',
            uplift_col: 'first'
        }).reset_index()

        # Квантильный разрез по uplift
        agg_sorted = agg.sort_values(by=uplift_col, ascending=False).reset_index(drop=True)
        total_len = len(agg_sorted)

        for q in quantiles:
            cutoff = int(q * total_len)
            subset = agg_sorted.iloc[:cutoff]
            n_clients = len(subset)

            treat = subset[subset['treatment_flg'] == 1]
            control = subset[subset['treatment_flg'] == 0]

            # Средние чеки
            treat_mean = treat['purchase_sum'].mean()
            control_mean = control['purchase_sum'].mean()
            uplift_rub = treat_mean - control_mean

            # Конверсии
            conv_treat = treat['target'].mean()
            conv_control = control['target'].mean()
            uplift_conv = conv_treat - conv_control

            # ROI: uplift в рублях — стоимость смс
            roi = uplift_rub - sms_cost

            results.append({
                'Модель': uplift_col,
                'Горизонт': horizon_name,
                'Доля базы': f"{int(q * 100)}%",
                'Кол-во клиентов': n_clients,
                'Конверсия TG': round(conv_treat, 4),
                'Конверсия CG': round(conv_control, 4),
                'Uplift конверсии': round(uplift_conv, 4),
                'Средний чек (TG)': round(treat_mean, 2),
                'Средний чек (CG)': round(control_mean, 2),
                'Инкремент (руб)': round(uplift_rub, 2),
                'ОП (инк. - СМС)': round(roi, 2),
            })

    return pd.DataFrame(results)

In [14]:
X_val['ct_uplift'] = ct_uplift
X_val['dml_uplift'] = dml_uplift
X_val = pd.concat([X_val, treat_val, y_val], axis=1)

In [15]:
X_val = X_val.merge(dataset.data['purchases'][['client_id', 'purchase_sum', 'transaction_datetime']], how='left', on=['client_id'])

In [21]:
dml_ct_full = build_full_roi_table(X_val, uplift_col='dml_uplift', campaign_date='2023-01-01', sms_cost=8)

In [28]:
final_summary = dml_ct_full.loc[
    ((dml_ct_full['Горизонт'] == '1w') & (dml_ct_full['Доля базы'] == '40%')) |
    ((dml_ct_full['Горизонт'] == '2m') & (dml_ct_full['Доля базы'] == '60%')) |
    ((dml_ct_full['Горизонт'] == '3m') & (dml_ct_full['Доля базы'] == '100%')) |
    ((dml_ct_full['Горизонт'] == '6m') & (dml_ct_full['Доля базы'] == '70%'))
].copy()

final_summary = final_summary[[
    'Горизонт', 'Доля базы', 'Кол-во клиентов',
    'Конверсия TG', 'Конверсия CG', 'Uplift конверсии',
    'Средний чек (TG)', 'ОП (инк. - СМС)'
]]

# Переименуем для презентабельности
final_summary.columns = [
    'Горизонт', 'Доля базы', 'N клиентов',
    'Конверсия TG', 'Конверсия CG', 'Uplift',
    'Средний чек (TG)', 'Инкремент ОП (₽)'
]

display(final_summary.sort_values(by=['Горизонт', 'N клиентов']))

,Горизонт,Доля базы,N клиентов,Конверсия TG,Конверсия CG,Uplift,Средний чек (TG),Инкремент ОП (₽)
0,1w,40%,24004,0.6666,0.6089,0.0577,81527.16,4236.31
9,2m,60%,36007,0.6698,0.6182,0.0516,93864.39,3549.28
20,3m,100%,60012,0.6370,0.6000,0.0370,89851.04,1449.55
24,6m,70%,42008,0.6641,0.6197,0.0444,93609.39,3631.30


In [29]:
def build_full_roi_table(
    df,
    uplift_col,
    campaign_date="2023-01-01",
    quantiles=(0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0),
    sms_cost=8
):
    df = df.copy()

    df["transaction_datetime"] = pd.to_datetime(df["transaction_datetime"])
    campaign_date = pd.to_datetime(campaign_date)

    df["days_since_campaign"] = (
        df["transaction_datetime"] - campaign_date
    ).dt.days

    # На всякий случай убираем покупки до даты старта
    df = df[df["days_since_campaign"] >= 0].copy()

    horizons = {
        "1w": 7,
        "2m": 60,
        "3m": 90,
        "6m": 180,
    }

    result_rows = []

    for horizon_name, max_days in horizons.items():

        # Берем только транзакции внутри горизонта
        horizon_df = df[df["days_since_campaign"] <= max_days].copy()

        if horizon_df.empty:
            continue

        # Один клиент — одна строка внутри горизонта
        agg_df = (
            horizon_df
            .groupby("client_id")
            .agg({
                "treatment_flg": "first",
                "target": "max",
                "purchase_sum": "sum",
                uplift_col: "first",
            })
            .reset_index()
        )

        # Сортируем клиентов по uplift-скору модели
        agg_df = agg_df.sort_values(by=uplift_col, ascending=False).reset_index(drop=True)

        total_n = len(agg_df)

        for q in quantiles:
            cutoff = int(total_n * q)
            subset = agg_df.iloc[:cutoff].copy()

            treat = subset[subset["treatment_flg"] == 1]
            control = subset[subset["treatment_flg"] == 0]

            conv_treat = treat["target"].mean()
            conv_control = control["target"].mean()
            uplift_conv = conv_treat - conv_control

            avg_check_treat = treat["purchase_sum"].mean()
            avg_check_control = control["purchase_sum"].mean()

            incr_rub = avg_check_treat - avg_check_control
            op_after_sms = incr_rub - sms_cost

            result_rows.append({
                "Модель": uplift_col,
                "Горизонт": horizon_name,
                "Доля базы": f"{int(q * 100)}%",
                "Кол-во клиентов": len(subset),

                "Конверсия в контр. группе": conv_control,
                "Конверсия в тестовой группе": conv_treat,
                "Инкремент конверсии": uplift_conv,

                "Средний чек (control)": avg_check_control,
                "Средний чек (treatment)": avg_check_treat,

                "Инкремент ОП на клиента, руб": op_after_sms,
            })

    return pd.DataFrame(result_rows)

In [31]:
dml_full = build_full_roi_table(
    df=X_val,
    uplift_col="dml_uplift",
    campaign_date="2023-01-01",
    quantiles=(0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0),
    sms_cost=8
)

dml_full

""


In [34]:
X_val.shape

(6856118, 12)

In [36]:
base_val = X_val.copy()

# Если client_id находится в индексе
if 'client_id' not in base_val.columns:
    base_val = base_val.reset_index()

base_val['ct_uplift'] = ct_uplift
base_val['dml_uplift'] = dml_uplift

# treatment и target тоже обычно приходят с индексом client_id
base_val = base_val.merge(
    treat_val.rename('treatment_flg').reset_index(),
    on='client_id',
    how='left'
)

base_val = base_val.merge(
    y_val.rename('target').reset_index(),
    on='client_id',
    how='left'
)

# Оставляем только одну строку на клиента
base_val = base_val.drop_duplicates(subset=['client_id']).copy()


# =========================
# 2. Покупки только для validation-клиентов
# =========================

purchases_val = dataset.data['purchases'][[
    'client_id',
    'transaction_datetime',
    'purchase_sum'
]].copy()

purchases_val['transaction_datetime'] = pd.to_datetime(
    purchases_val['transaction_datetime']
)

purchases_val = purchases_val[
    purchases_val['client_id'].isin(base_val['client_id'])
].copy()

In [37]:
def build_business_case_long(
    base_df,
    purchases_df,
    uplift_col,
    campaign_date=None,
    quantiles=(0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0),
    sms_cost=8
):
    """
    Возвращает длинную таблицу:
    модель × горизонт × доля базы.

    Важно:
    - base_df: один client_id = одна строка;
    - purchases_df: много покупок на клиента;
    - N клиентов одинаковый для всех горизонтов при одной доле базы;
    - purchase_sum считается внутри каждого горизонта;
    - target_horizon = была ли покупка внутри горизонта.
    """

    base_df = base_df.copy()
    purchases_df = purchases_df.copy()

    purchases_df['transaction_datetime'] = pd.to_datetime(
        purchases_df['transaction_datetime']
    )

    # Если campaign_date не задана — берем самую раннюю дату покупки в validation
    if campaign_date is None:
        campaign_date = purchases_df['transaction_datetime'].min()
    else:
        campaign_date = pd.to_datetime(campaign_date)

    purchases_df['days_since_campaign'] = (
        purchases_df['transaction_datetime'] - campaign_date
    ).dt.days

    purchases_df = purchases_df[purchases_df['days_since_campaign'] >= 0].copy()

    horizons = {
        '1w': 7,
        '2m': 60,
        '3m': 90,
        '6m': 180,
    }

    rows = []

    # Клиентов сортируем один раз по скору модели
    base_sorted = base_df.sort_values(by=uplift_col, ascending=False).reset_index(drop=True)
    total_n = len(base_sorted)

    for q in quantiles:
        cutoff = int(total_n * q)
        top_clients = base_sorted.iloc[:cutoff].copy()

        client_ids = top_clients['client_id']

        for horizon_name, max_days in horizons.items():

            horizon_purchases = purchases_df[
                (purchases_df['client_id'].isin(client_ids)) &
                (purchases_df['days_since_campaign'] <= max_days)
            ].copy()

            # Агрегируем покупки по клиенту внутри горизонта
            purchase_agg = (
                horizon_purchases
                .groupby('client_id')
                .agg(purchase_sum_horizon=('purchase_sum', 'sum'))
                .reset_index()
            )

            horizon_df = top_clients.merge(
                purchase_agg,
                on='client_id',
                how='left'
            )

            # Клиенты без покупок внутри горизонта получают 0
            horizon_df['purchase_sum_horizon'] = horizon_df['purchase_sum_horizon'].fillna(0)

            # Конверсия внутри горизонта
            horizon_df['target_horizon'] = (
                horizon_df['purchase_sum_horizon'] > 0
            ).astype(int)

            treat = horizon_df[horizon_df['treatment_flg'] == 1]
            control = horizon_df[horizon_df['treatment_flg'] == 0]

            conv_treat = treat['target_horizon'].mean()
            conv_control = control['target_horizon'].mean()
            uplift_conv = conv_treat - conv_control

            avg_check_treat = treat['purchase_sum_horizon'].mean()
            avg_check_control = control['purchase_sum_horizon'].mean()

            incr_rub = avg_check_treat - avg_check_control
            op_after_sms = incr_rub - sms_cost

            rows.append({
                'Модель': uplift_col,
                'Горизонт': horizon_name,
                'Доля базы': f'{int(q * 100)}%',
                'Кол-во наблюдений': len(horizon_df),

                'Конверсия в контр. группе': conv_control,
                'Конверсия в тестовой группе': conv_treat,
                'Инкремент конверсии': uplift_conv,

                'Средний чек в контр. группе': avg_check_control,
                'Средний чек в тестовой группе': avg_check_treat,

                'Инкремент ОП на клиента, руб': op_after_sms,
            })

    return pd.DataFrame(rows)

In [38]:
def make_burger_king_table(long_df):
    """
    Превращает длинную таблицу в широкий формат:
    доля базы + N + блоки по горизонтам.
    """

    long_df = long_df.copy()

    horizon_order = ['1w', '2m', '3m', '6m']

    horizon_titles = {
        '1w': 'Покупки на горизонте недели',
        '2m': 'Покупки на горизонте 2 месяцев',
        '3m': 'Покупки на горизонте 3 месяцев',
        '6m': 'Покупки на горизонте полугода',
    }

    # Форматирование конверсии control
    long_df['conv_control_fmt'] = (
        (long_df['Конверсия в контр. группе'] * 100)
        .round(1)
        .astype(str)
        .str.replace('.', ',', regex=False)
        + '%'
    )

    # Форматирование конверсии treatment + прирост
    long_df['conv_treat_fmt'] = (
        (long_df['Конверсия в тестовой группе'] * 100)
        .round(1)
        .astype(str)
        .str.replace('.', ',', regex=False)
        + '% '
        + long_df['Инкремент конверсии'].apply(
            lambda x: f"(+{x * 100:.1f}%)" if x >= 0 else f"({x * 100:.1f}%)"
        ).str.replace('.', ',', regex=False)
    )

    # Инкремент ОП
    long_df['op_fmt'] = (
        long_df['Инкремент ОП на клиента, руб']
        .round(1)
    )

    # База строк
    final = (
        long_df[['Доля базы', 'Кол-во наблюдений']]
        .drop_duplicates()
        .copy()
    )

    final['_sort'] = final['Доля базы'].str.replace('%', '').astype(int)
    final = final.sort_values('_sort').drop(columns='_sort').reset_index(drop=True)

    # Добавляем блоки горизонтов
    for h in horizon_order:
        h_df = long_df[long_df['Горизонт'] == h].copy()

        h_df = h_df[[
            'Доля базы',
            'conv_control_fmt',
            'conv_treat_fmt',
            'op_fmt'
        ]].rename(columns={
            'conv_control_fmt': f'{horizon_titles[h]} | Конверсия в контр. группе',
            'conv_treat_fmt': f'{horizon_titles[h]} | Конверсия в тестовой группе',
            'op_fmt': f'{horizon_titles[h]} | Инкремент ОП на клиента, руб',
        })

        final = final.merge(h_df, on='Доля базы', how='left')

    return final

In [39]:
dml_long = build_business_case_long(
    base_df=base_val,
    purchases_df=purchases_val,
    uplift_col='dml_uplift',
    campaign_date=None,  # автоматически возьмет минимальную дату покупки
    quantiles=(0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0),
    sms_cost=8
)

dml_burger_king_table = make_burger_king_table(dml_long)

dml_burger_king_table

,Доля базы,Кол-во наблюдений,Покупки на горизонте недели | Конверсия в контр. группе,Покупки на горизонте недели | Конверсия в тестовой группе,"Покупки на горизонте недели | Инкремент ОП на клиента, руб",Покупки на горизонте 2 месяцев | Конверсия в контр. группе,Покупки на горизонте 2 месяцев | Конверсия в тестовой группе,"Покупки на горизонте 2 месяцев | Инкремент ОП на клиента, руб",Покупки на горизонте 3 месяцев | Конверсия в контр. группе,Покупки на горизонте 3 месяцев | Конверсия в тестовой группе,"Покупки на горизонте 3 месяцев | Инкремент ОП на клиента, руб",Покупки на горизонте полугода | Конверсия в контр. группе,Покупки на горизонте полугода | Конверсия в тестовой группе,"Покупки на горизонте полугода | Инкремент ОП на клиента, руб"
0,40%,24004,"55,2%","53,4% (-1,8%)",48.4,"90,9%","90,2% (-0,7%)",1568.6,"95,6%","95,6% (-0,1%)",3234.8,"100,0%","100,0% (+0,0%)",4109.3
1,50%,30006,"56,9%","55,7% (-1,2%)",118.1,"91,8%","91,5% (-0,3%)",1800.8,"96,0%","96,1% (+0,1%)",3277.6,"100,0%","100,0% (+0,0%)",4094.3
2,60%,36007,"58,0%","57,1% (-0,9%)",98.2,"92,5%","92,2% (-0,2%)",1808.5,"96,4%","96,5% (+0,1%)",2857.8,"100,0%","100,0% (+0,0%)",3435.8
3,70%,42008,"57,0%","56,2% (-0,7%)",199.7,"91,2%","91,4% (+0,1%)",2244.3,"96,0%","96,2% (+0,2%)",3185.6,"100,0%","100,0% (+0,0%)",3873.8
4,80%,48009,"56,2%","55,7% (-0,4%)",125.3,"91,0%","91,2% (+0,1%)",1886.9,"95,9%","96,2% (+0,2%)",2555.5,"100,0%","100,0% (+0,0%)",2893.1
5,90%,54010,"55,7%","55,4% (-0,3%)",73.6,"90,6%","90,8% (+0,2%)",1300.9,"95,7%","96,0% (+0,3%)",1647.0,"100,0%","100,0% (+0,0%)",1587.5
6,100%,60012,"53,3%","53,4% (+0,1%)",55.5,"89,3%","89,8% (+0,5%)",1181.1,"95,1%","95,5% (+0,4%)",1456.6,"100,0%","100,0% (+0,0%)",1449.5


In [41]:
! pip install openpyxl

In [44]:
dml_burger_king_table.to_pickle('res.pkl')